## Anomaly Detection in Aadhaar Update Activity (2025)

This notebook identifies abnormal spikes in Aadhaar biometric and demographic updates across Indian states and districts for the year 2025.

Anomaly detection is performed at a monthly granularity, using multiple complementary techniques:

- Statistical Z-score
- Interquartile Range (IQR)
- Machine-learning-based Isolation Forest

The analysis builds upon the statistical foundations established in the EDA phase.

In [1]:
#!pip install scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

In [3]:
biometric = pd.read_csv("../data/processed/biometric_clean.csv")
demographic = pd.read_csv("../data/processed/demographic_clean.csv")

In [4]:
biometric["bio_total"] = (
    biometric["bio_age_5_17"] +
    biometric["bio_age_17_"]
)

demographic["demo_total"] = (
    demographic["demo_age_5_17"] +
    demographic["demo_age_17_"]
)

In [5]:
biometric["date"] = pd.to_datetime(biometric["date"], dayfirst=True)
demographic["date"] = pd.to_datetime(demographic["date"], dayfirst=True)

biometric["month"] = biometric["date"].dt.month
demographic["month"] = demographic["date"].dt.month

In [6]:
bio_state_month = (
    biometric
    .groupby(["state", "month"])["bio_total"]
    .sum()
    .reset_index()
)

demo_state_month = (
    demographic
    .groupby(["state", "month"])["demo_total"]
    .sum()
    .reset_index()
)

METHOD 1 — Z-SCORE ANOMALY DETECTION

In [7]:
def safe_zscore(x):
    if x.std() == 0:
        return np.zeros(len(x))
    return (x - x.mean()) / x.std()

bio_state_month["z_score"] = (
    bio_state_month
    .groupby("state")["bio_total"]
    .transform(safe_zscore)
)

In [8]:
bio_state_month["z_anomaly"] = bio_state_month["z_score"].abs() > 3

# Check anomalies:
bio_state_month[bio_state_month["z_anomaly"]].sort_values(
    "z_score", ascending=False
)

,state,month,bio_total,z_score,z_anomaly


|Z| > 3 → statistically extreme spike

METHOD 2 — IQR-BASED ANOMALY DETECTION

In [9]:
def detect_iqr_anomalies(df, value_col):
    q1 = df[value_col].quantile(0.25)
    q3 = df[value_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (df[value_col] < lower) | (df[value_col] > upper)

In [10]:
bio_state_month["iqr_anomaly"] = (
    bio_state_month
    .groupby("state", group_keys=False)
    .apply(lambda x: detect_iqr_anomalies(x, "bio_total"))
)

# Check:
bio_state_month[bio_state_month["iqr_anomaly"]]

C:\Users\mrark\AppData\Local\Temp\ipykernel_3376\1136334811.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: detect_iqr_anomalies(x, "bio_total"))


,state,month,bio_total,z_score,z_anomaly,iqr_anomaly
49,Chandigarh,7,32547,2.623218,False,True
69,Dadra and Nagar Haveli and Daman and Diu,10,1302,-2.060752,False,True
86,Goa,9,4692,-1.645755,False,True
103,Haryana,7,267325,1.862605,False,True
186,Maharashtra,10,690606,-1.899164,False,True
193,Manipur,7,86754,2.219316,False,True
196,Manipur,11,57408,1.058356,False,True
207,Mizoram,3,51069,2.448539,False,True
208,Mizoram,4,24235,0.711668,False,True
239,Puducherry,9,12832,2.218734,False,True


METHOD 3 — ISOLATION FOREST (ML)

In [11]:
iso_df = np.log1p(bio_state_month[["bio_total"]])

In [12]:
iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.015,
    random_state=42
)

bio_state_month["if_anomaly"] = iso_model.fit_predict(iso_df)

# Convert to boolean:
bio_state_month["if_anomaly"] = bio_state_month["if_anomaly"] == -1

In [13]:
bio_state_month["combined_anomaly"] = (
    bio_state_month["z_anomaly"] |
    bio_state_month["iqr_anomaly"] |
    bio_state_month["if_anomaly"]
)

# Check strongest cases:
bio_state_month[bio_state_month["combined_anomaly"]]

,state,month,bio_total,z_score,z_anomaly,iqr_anomaly,if_anomaly,combined_anomaly
49,Chandigarh,7,32547,2.623218,False,True,False,True
69,Dadra and Nagar Haveli and Daman and Diu,10,1302,-2.060752,False,True,False,True
86,Goa,9,4692,-1.645755,False,True,False,True
103,Haryana,7,267325,1.862605,False,True,False,True
156,Ladakh,6,339,-1.241568,False,False,True,True
167,Lakshadweep,9,328,-1.520251,False,False,True,True
168,Lakshadweep,10,453,-0.566385,False,False,True,True
186,Maharashtra,10,690606,-1.899164,False,True,False,True
193,Manipur,7,86754,2.219316,False,True,False,True
196,Manipur,11,57408,1.058356,False,True,False,True


In [14]:
bio_district_month = (
    biometric
    .groupby(["state", "district", "month"])["bio_total"]
    .sum()
    .reset_index()
)

# Z-score district-wise:
bio_district_month["z_score"] = (
    bio_district_month
    .groupby("district")["bio_total"]
    .transform(lambda x: (x - x.mean()) / x.std())
)

bio_district_month["z_anomaly"] = bio_district_month["z_score"].abs() > 3

In [15]:
anomaly_summary = bio_state_month[
    bio_state_month["combined_anomaly"]
].assign(
    severity=lambda x: x["z_score"].abs()
).sort_values("severity", ascending=False)

anomaly_summary.head(20)

# Save:
anomaly_summary.to_csv(
    "../outputs/tables/state_month_anomalies.csv",
    index=False
)

## Anomaly Detection Insights

Multiple states exhibit statistically significant monthly spikes in biometric updates.

These spikes are consistently identified across statistical (Z-score, IQR) and machine-learning (Isolation Forest) methods.

Such anomalies may indicate operational surges, biometric ageing effects, targeted enrolment drives, or system stress periods.

These findings provide actionable inputs for proactive capacity planning and monitoring.

## Methodological Note

Z-score calculations were stabilised for low-variance states, and Isolation Forest was applied on log-transformed data to mitigate scale effects.

The use of multiple complementary anomaly detection methods ensures robustness and reduces false positives.